# TerraDS Deep Probe — Phase 2, Step 1

The first explorer showed TerraDS stores **extracted metadata**, not raw `.tf` files.
The key column is `Modules.ModuleCalls` (cross-module dependencies) and `Modules.Providers`.
This probe answers: how often are these populated, what do they look like, and how do
the tables join — everything needed to build the dependency-graph extractor.

Run all, then paste back the full output.

In [ ]:
import sqlite3, json, glob, os
from collections import Counter

DB = "/content/terrads/TerraDS.sqlite"
if not os.path.exists(DB):
    hits = glob.glob("/content/**/TerraDS.sqlite", recursive=True) + glob.glob("**/TerraDS.sqlite", recursive=True)
    DB = hits[0] if hits else DB
print("Using DB:", DB)
con = sqlite3.connect(DB); cur = con.cursor()

# ---- Q1: how many modules have NON-EMPTY ModuleCalls / Providers? ----
total = cur.execute("SELECT COUNT(*) FROM Modules").fetchone()[0]
nonempty_calls = cur.execute(
    "SELECT COUNT(*) FROM Modules WHERE ModuleCalls IS NOT NULL AND ModuleCalls != '[]' AND ModuleCalls != ''").fetchone()[0]
nonempty_prov = cur.execute(
    "SELECT COUNT(*) FROM Modules WHERE Providers IS NOT NULL AND Providers != '[]' AND Providers != ''").fetchone()[0]
print(f"\nModules total: {total}")
print(f"  with non-empty ModuleCalls: {nonempty_calls}  ({100*nonempty_calls/total:.1f}%)")
print(f"  with non-empty Providers:   {nonempty_prov}  ({100*nonempty_prov/total:.1f}%)")

# ---- Q2: show 8 real non-empty ModuleCalls values (the dependency structure) ----
print("\n--- Sample NON-EMPTY ModuleCalls (up to 8) ---")
rows = cur.execute(
    "SELECT Id, RepositoryId, Path, ModuleCalls FROM Modules "
    "WHERE ModuleCalls IS NOT NULL AND ModuleCalls != '[]' AND ModuleCalls != '' LIMIT 8").fetchall()
for r in rows:
    print(f"\nModuleId={r[0]} RepoId={r[1]} Path={r[2]}")
    try:
        parsed = json.loads(r[3])
        print("  parsed ModuleCalls:", json.dumps(parsed, indent=2)[:600])
    except Exception as e:
        print("  raw:", str(r[3])[:400])

# ---- Q3: distribution of modules per repository (cross-module surface) ----
print("\n--- Modules per repository (distribution) ---")
counts = [r[0] for r in cur.execute(
    "SELECT COUNT(*) c FROM Modules GROUP BY RepositoryId").fetchall()]
counts.sort()
import statistics as st
print(f"  repos with >=1 module: {len(counts)}")
print(f"  min={min(counts)} median={st.median(counts)} mean={st.mean(counts):.1f} max={max(counts)}")
buckets = Counter()
for c in counts:
    b = "1" if c==1 else "2-3" if c<=3 else "4-10" if c<=10 else "11-50" if c<=50 else "50+"
    buckets[b]+=1
for b in ["1","2-3","4-10","11-50","50+"]:
    print(f"    {b:6s} modules: {buckets[b]} repos")

# ---- Q4: sample Providers values ----
print("\n--- Sample non-empty Providers (up to 8) ---")
for r in cur.execute("SELECT DISTINCT Providers FROM Modules WHERE Providers != '[]' LIMIT 8").fetchall():
    print("  ", r[0])

# ---- Q5: resource type frequency (top 15) ----
print("\n--- Top 15 resource types ---")
for t,c in cur.execute("SELECT Type, COUNT(*) c FROM Resources GROUP BY Type ORDER BY c DESC LIMIT 15").fetchall():
    print(f"  {c:8d}  {t}")

# ---- Q6: repos where IaC is a SMALL part (your gap) — proxy via SizeInKb vs modules ----
print("\n--- Repo size distribution (SizeInKb) — proxy for 'IaC is small part' ---")
sizes = [r[0] for r in cur.execute("SELECT SizeInKb FROM Repositories WHERE SizeInKb IS NOT NULL").fetchall()]
sizes.sort()
if sizes:
    print(f"  min={min(sizes)} median={st.median(sizes)} mean={st.mean(sizes):.0f} max={max(sizes)}")

con.close()
print("\n>>> DONE. Paste all output back. <<<")